# Frequency Counting

---

## What is Frequency Counting?

Frequency counting is a pattern where you use a hash map to **record how many times each item appears**.

```
item  →  count
```

That's it. The hash map's key is the item, and the value is how many times you've seen it.

### Why does this matter?

Frequency counting is not just a coding trick — it's the foundation of **how AI understands text**:

- **TF-IDF** (Term Frequency-Inverse Document Frequency) — counts word frequencies to find what's important in a document
- **Bag of Words** — represents a document as a frequency map of its words
- **Tokenizer vocabularies** — LLMs are trained on frequency distributions of tokens
- **Recommendation systems** — count how often a user interacts with each category

Every time you use ChatGPT, the model underneath was trained on frequency-counted data at massive scale.

---
## Part 1 — The Core Pattern

There are two common ways to build a frequency map. Know both.

**Method 1: `dict.get(key, default)`** — no imports needed
```python
freq = {}
for item in collection:
    freq[item] = freq.get(item, 0) + 1
```

**Method 2: `collections.Counter`** — cleaner, same result
```python
from collections import Counter
freq = Counter(collection)
```

Both produce the same dict-like object. In interviews, `Counter` is fine to use — it shows you know the standard library. But understanding Method 1 is essential because you'll extend it in ways `Counter` alone can't handle.

In [2]:
# build a frequency map manually, then verify using Counter
# given this list of words, count how many times each word appears

from collections import Counter

words = ["apple", "banana", "apple", "cherry", "banana", "apple"]

# Method 1: manual
freq_manual = {}
for word in words:
    freq_manual[word] = freq_manual.get(word, 0) + 1

print("Manual:", freq_manual)

# Method 2: Counter
freq_counter = Counter(words)
print("Counter:", freq_counter)
print(type(freq_counter))

# they should be equal
print("Equal:", dict(freq_counter) == freq_manual)

Manual: {'apple': 3, 'banana': 2, 'cherry': 1}
Counter: Counter({'apple': 3, 'banana': 2, 'cherry': 1})
<class 'collections.Counter'>
Equal: True


# Regex (Regular Expressions)

A mini-language for describing patterns in text. Instead of saying "find the word cat",
you describe the **shape** of what you're looking for.

---

## Special Characters

| Character | Meaning | Example |
|-----------|---------|---------|
| `.` | any single character | `c.t` matches `cat`, `cot`, `cut` |
| `*` | 0 or more of the previous | `ca*t` matches `ct`, `cat`, `caat` |
| `+` | 1 or more of the previous | `ca+t` matches `cat`, `caat` but NOT `ct` |
| `?` | 0 or 1 of the previous | `ca?t` matches `ct` or `cat` only |
| `^` | start of string | `^the` matches only if string starts with "the" |
| `$` | end of string | `end$` matches only if string ends with "end" |

---

## Character Classes `[ ]`

```python
[abc]      # matches a, b, or c
[a-z]      # any lowercase letter
[A-Z]      # any uppercase letter
[0-9]      # any digit
[a-zA-Z]   # any letter
[a-zA-Z ]  # any letter OR space
[^0-9]     # anything NOT a digit  (^ inside brackets means NOT)
[^a-zA-Z ] # anything NOT a letter or space

## find and replace - re.sub

re.sub(pattern, replacement, string)
# finds every match of pattern and replaces it with replacement

re.sub(r"[0-9]", "*", "my pin is 4821")
# → "my pin is ****"

re.sub(r"[^a-zA-Z ]", "", "Hello, World! 123")
# → "Hello World "

## other useful functions
re.search(pattern, string)   # find first match anywhere in string
re.match(pattern, string)    # match only at the START of string
re.findall(pattern, string)  # return list of ALL matches
re.split(pattern, string)    # split string on the pattern

## shorthand classes
\d   # same as [0-9]        — digit
\w   # same as [a-zA-Z0-9_] — word character
\s   # whitespace (space, tab, newline)
\D   # NOT a digit   (uppercase = opposite)
\W   # NOT a word character
\S   # NOT whitespace


re.findall(r"[0-9]+", "order 42 costs $19.99")  # → ["42", "19", "99"]
re.split(r"\s+", "hello   world  foo")           # → ["hello", "world", "foo"]



---
## Part 2 — Word Frequency Analyzer

Now apply frequency counting to real text. This is exactly what NLP pipelines do.

A production word frequency analyzer needs to:
1. **Normalize** the text (lowercase, strip punctuation) — otherwise "Apple" and "apple" are counted as different words
2. **Tokenize** — split into individual words
3. **Count** — build the frequency map
4. **Rank** — find the most/least common words

Step 4 is where another hash map pattern comes in: **sorting by value**, not by key.



In [4]:
import re

text = """
To be or not to be, that is the question.
Whether 'tis nobler in the mind to suffer
the slings and arrows of outrageous fortune,
or to take arms against a sea of troubles.
"""

# step 1 — normalize: lowercase, remove punctuation
# re.sub replaces all non-letter, non-space characters with empty string
# re.sub(pattern, replacement, string)
lower_case = text.lower()
cleaned = re.sub(r"[^a-zA-Z\s]", "", lower_case)

# step 2 — tokenize
## ---------  split(" ") vs split()----------
# split(" ") splits only on a single literal space. If there are multiple consecutive spaces, you get empty strings in the list:
# "hello  world".split(" ")   # → ["hello", "", "world"]  ← empty string!
# "hello  world".split()      # → ["hello", "world"]       ← clean
# split() with no argument splits on any whitespace and discards empties. Always prefer it for tokenizing.

words = cleaned.split()

# step 3 — build frequency map (use manual method here)
freq = {}
for word in words:
    freq[word] = freq.get(word, 0) + 1


# step 4 — top 5 most common words
# sorted() returns a list; key= tells it how to compare items
# lambda item: item[1]  means "compare by the value (count), not the key (word)"
# reverse=True means highest count first
top_5 = sorted(freq.items(), key = lambda item: item[1], reverse=True)[:5]

print("Word frequencies:", freq)
print("\nTop 5 most common:", top_5)

Word frequencies: {'to': 4, 'be': 2, 'or': 2, 'not': 1, 'that': 1, 'is': 1, 'the': 3, 'question': 1, 'whether': 1, 'tis': 1, 'nobler': 1, 'in': 1, 'mind': 1, 'suffer': 1, 'slings': 1, 'and': 1, 'arrows': 1, 'of': 2, 'outrageous': 1, 'fortune': 1, 'take': 1, 'arms': 1, 'against': 1, 'a': 1, 'sea': 1, 'troubles': 1}

Top 5 most common: [('to', 4), ('the', 3), ('be', 2), ('or', 2), ('of', 2)]


### How `sorted` with `key=` works

`freq.items()` gives you pairs like `[("the", 3), ("to", 4), ("or", 2)]`.

Without a key, `sorted` compares tuples left-to-right — it would sort alphabetically by word.

With `key=lambda item: item[1]`, you're saying: *"when comparing two items, only look at index 1 (the count)"*.

```python
# these do the same thing
sorted(freq.items(), key=lambda item: item[1], reverse=True)
sorted(freq.items(), key=lambda kv: kv[1], reverse=True)   # kv = key-value

# Counter has a shortcut for this exact pattern
Counter(words).most_common(5)  # same result
```

---
## Part 3 — Grouping Pattern

Frequency counting's sibling is **grouping**: instead of counting items, you **collect** them under a shared key.

```
grouped_key  →  [list of items that share this key]
```

The pattern is almost identical to frequency counting — the only difference is you append to a list instead of incrementing a number:

```python
# frequency counting        →    grouping
freq[key] = freq.get(key, 0) + 1    groups[key] = groups.get(key, []) + [item]

# or with setdefault (cleaner for lists):
groups.setdefault(key, []).append(item)
```

### Where grouping appears in ML:
- **Groupby in Pandas** — groups rows by a category column, then aggregates
- **Batch processing** — group documents by type before embedding
- **Label encoding** — group samples by class label
- **Group Anagrams** (coming up in LeetCode) — group words by their sorted character signature

In [ ]:
# practice the grouping pattern
# group these words by their FIRST LETTER
# example: "apple" and "avocado" both go under key "a"

words = ["apple", "banana", "avocado", "blueberry", "cherry", "apricot", "cantaloupe"]

# use setdefault — it's the cleanest way to build groups
# dict.setdefault(key, default) returns the value at key if it exists,
# otherwise sets key to default AND returns that default

groups = {}


print(groups)
# expected: {'a': ['apple', 'avocado', 'apricot'], 'b': ['banana', 'blueberry'], ...}

---
## Part 4 — TF-IDF Intuition (NLP Foundation)

**TF-IDF** is the most important classical NLP technique — it converts raw text into numbers that ML models can use. Every search engine, recommendation system, and document classifier has used some form of it.

It answers: *"which words in this document are actually meaningful?"*

### The problem with raw frequency

If you just count word frequencies in a news article, the most common words will be "the", "a", "is", "and" — they appear everywhere and tell you nothing about the topic.

### The two-part fix

**TF (Term Frequency)** — how often does this word appear in *this specific document*?
```
TF(word, doc) = count of word in doc / total words in doc
```

**IDF (Inverse Document Frequency)** — how rare is this word *across all documents*?  
Words like "the" appear in every document → low IDF (common = less informative).  
Words like "photosynthesis" appear in few documents → high IDF (rare = more informative).
```
IDF(word) = log(total docs / docs containing word)
```

**TF-IDF = TF × IDF** — high score means: frequent in this doc, rare across all docs → important signal.

The hash map at the core of TF-IDF is exactly the frequency map you just built.

In [8]:
import math

# implement a minimal TF-IDF from scratch using only frequency maps
# no sklearn — just dicts and math

docs = [
    "the cat sat on the mat",
    "the cat chased the dog",
    "the dog barked at the moon",
]

def term_frequency(word, doc_words):
    # count of word in doc_words / total words in doc_words
    freq = {}
    for w in doc_words:
        freq[w] = freq.get(w, 0) + 1
    tf = freq.get(word, 0) / len(doc_words)
    return tf

def inverse_document_frequency(word, all_docs):
    # log(total docs / number of docs that contain word)
    # return 0 if no doc contains the word
    docs_with_word = sum(1 for doc in all_docs if word in doc.split())
    
    if docs_with_word == 0:
        return 0
    idf = math.log (len(all_docs) / docs_with_word)
    return idf

def tfidf(word, doc, all_docs):
    doc_words = doc.split()
    tf = term_frequency(word, doc_words)
    idf = inverse_document_frequency(word, all_docs)
    return tf * idf

# compare TF-IDF scores for "the" vs "cat" in the first document
doc_0 = docs[0]
print(f'TF-IDF of "the"  in doc 0: {tfidf("the",  doc_0, docs):.4f}')   # should be low — common word
print(f'TF-IDF of "cat"  in doc 0: {tfidf("cat",  doc_0, docs):.4f}')   # should be higher — less common
print(f'TF-IDF of "moon" in doc 0: {tfidf("moon", doc_0, docs):.4f}')   # should be 0 — not in doc 0
print(f'TF-IDF of "moon" in doc 2: {tfidf("moon", docs[2], docs):.4f}') # should be highest — unique word

TF-IDF of "the"  in doc 0: 0.0000
TF-IDF of "cat"  in doc 0: 0.0676
TF-IDF of "moon" in doc 0: 0.0000
TF-IDF of "moon" in doc 2: 0.1831


## TF-IDF with sklearn

```python
from sklearn.feature_extraction.text import TfidfVectorizer

docs = [
    "the cat sat on the mat",
    "the cat chased the dog",
    "the dog barked at the moon",
]

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(docs)



### Reading the output

```python
import pandas as pd
df = pd.DataFrame(
    tfidf_matrix.toarray(),                     # sparse matrix → numpy array
    columns=vectorizer.get_feature_names_out()  # column per word
)
print(df.round(3))
```

- Each **row** = one document
- Each **column** = one word
- Each **cell** = TF-IDF score
- **High score** = frequent in this doc AND rare across all docs

---

### Useful parameters

```python
# Ignore words appearing in more than 85% of docs (e.g. "the")
TfidfVectorizer(max_df=0.85)

# Ignore words appearing in fewer than 2 docs (rare noise / typos)
TfidfVectorizer(min_df=2)

# Both together — standard production setup
TfidfVectorizer(max_df=0.85, min_df=2)
```

---

### Why `sklearn` over manual

| | Manual | `sklearn` |
|---|---|---|
| **Purpose** | Understand internals | Production use |
| **Speed** | Slow on large data | Optimized, sparse matrices |
| **Features** | Bare minimum | Stopwords, n-grams, normalization built in |

> **Rule of thumb:** Use `sklearn` in real pipelines. Build manually once so you understand what it's doing.

## Generator expressions — `sum(1 for ... if ...)`

A generator expression is a compact loop with an optional filter, written inline.

---

### The pattern

```python
sum(1 for item in collection if condition)
```

This **counts** how many items in `collection` satisfy `condition`.

---

### Breakdown: `sum(1 for doc in all_docs if word in doc.split())`

**Step 1 — `doc.split()`**  
Splits the string into a word list so you check whole words, not substrings.

```python
"the cat sat on the mat".split()
# → ["the", "cat", "sat", "on", "the", "mat"]
```

**Step 2 — `if word in doc.split()`**  
Checks if the word exists in that doc's word list.

```python
"cat" in ["the", "cat", "sat", "on", "the", "mat"]     # → True
"cat" in ["the", "dog", "barked", "at", "the", "moon"]  # → False
```

**Step 3 — `1 for doc in all_docs if ...`**  
Yields `1` only when the condition is `True`, skips otherwise.

```python
# doc 0: "cat" found → yield 1
# doc 1: "cat" found → yield 1
# doc 2: "cat" not found → skip
# produces: [1, 1]
```

**Step 4 — `sum(...)`**  
Adds up all the `1`s → count of matching docs.

```python
sum([1, 1])  # → 2
```

---

### Unrolled equivalent

```python
docs_with_word = 0
for doc in all_docs:
    if word in doc.split():
        docs_with_word += 1
```

The generator expression is just this written in one line.

---

### Why `doc.split()` instead of `word in doc`

```python
word = "cat"

# WRONG — substring check, matches "concatenate" too
"cat" in "the concatenate method"          # → True  ✗

# CORRECT — whole word check
"cat" in "the concatenate method".split()  # → False ✓
```

> Always split before checking membership when you want whole words.

---
## Part 5 — Two Patterns Side by Side

At this point you've seen two related patterns. Recognizing which one to use is the real skill:

| Pattern | Hash map value | Question it answers | Example |
|---------|---------------|--------------------|---------|
| **Frequency counting** | `int` (a count) | "How many times does X appear?" | word count, Top K frequent |
| **Grouping** | `list` (a collection) | "Which items share property X?" | group anagrams, groupby |

Both start with the same loop:
```python
for item in collection:
    key = some_function(item)   # compute the grouping key
    # then either:
    freq[key] = freq.get(key, 0) + 1        # counting
    groups.setdefault(key, []).append(item)  # grouping
```

The only difference is what you store as the value.

## Hash map skeleton — two patterns

Every time you loop through a collection with a hash map, you're asking one of two questions:

| Question | Pattern | Value type |
|---|---|---|
| "How many times?" | Frequency counting | `int` |
| "Which ones?" | Grouping | `list` |

---

### Frequency counting

```python
freq = {}
for word in words:
    freq[word] = freq.get(word, 0) + 1
# {"cat": 3, "dog": 2, "bird": 1}
```

---

### Grouping

```python
groups = {}
for word in words:
    key = word[0]                           # first letter
    groups.setdefault(key, []).append(word)
# {"c": ["cat", "cat", "cat"], "d": ["dog", "dog"], "b": ["bird"]}
```

---

### How to decide

- *"how many / frequency / count"* → frequency counting, value is `int`
- *"group by / which ones / find all"* → grouping, value is `list`

---

### `setdefault` explained

```python
groups.setdefault(key, []).append(item)
```

- Key **exists** → returns current value, appends to it
- Key **missing** → creates it with `[]`, then appends

Why not `get`?

```python
groups.get(key, []).append(item)  # WRONG — new [] is never stored
```

`get` returns the default but doesn't save it. `setdefault` saves it and returns it.

Manual equivalent:

```python
if key not in groups:
    groups[key] = []
groups[key].append(item)
```

---
## LeetCode

Try each problem for at least 20 minutes before asking for hints.

- [ ] Group Anagrams (Medium)
- [ ] Top K Frequent Elements (Medium)

In [11]:
# GROUP ANAGRAMS
# given a list of strings, group the anagrams together
# example: ["eat","tea","tan","ate","nat","bat"]
# → [["bat"],["nat","tan"],["ate","eat","tea"]]
# order of groups and order within groups doesn't matter
#
# KEY INSIGHT: two words are anagrams if and only if they have the same characters.
# what single operation turns any anagram into the same canonical form?
# hint: "eat", "tea", "ate" — sort each one alphabetically...

def group_anagrams(strs):
    groups = {}
    for word in strs:
        key = "".join(sorted(word))
        groups.setdefault(key, []).append(word)
    return list(groups.values())


result = group_anagrams(["eat","tea","tan","ate","nat","bat"])
print(result)
# expected: [["bat"],["nat","tan"],["ate","eat","tea"]] (any order)

[['eat', 'tea', 'ate'], ['tan', 'nat'], ['bat']]


### time complexity
O(n * k log k) time — n words, each sorted in k log k where k is word length

In [13]:
# TOP K FREQUENT ELEMENTS
# given a list of numbers and an integer k, return the k most frequent elements
# example: nums=[1,1,1,2,2,3], k=2 → [1, 2]
# the answer is guaranteed to be unique (no ties to break)
#
# approach: two steps you already know
# 1. build a frequency map
# 2. find the top k items by frequency

def top_k_frequent(nums, k):
    freq = {}
    for num in nums:
        freq[num] = freq.get(num, 0) + 1
    return sorted(freq, key=lambda num: freq[num], reverse=True)[:k]

print(top_k_frequent([1,1,1,2,2,3], 2))   # [1, 2]
print(top_k_frequent([1], 1))              # [1]

[1, 2]
[1]


## Sorting a frequency map — `sorted(freq, key=lambda num: freq[num])`

```python
nums = [1,1,1,2,2,3]
freq = {1: 3, 2: 2, 3: 1}

sorted(freq, key=lambda num: freq[num], reverse=True)[:k]
```

---

### Step 1 — `sorted(freq, ...)` iterates over keys

Passing a dict to `sorted()` gives you the keys, not the values:

```python
sorted(freq)  # → [1, 2, 3]
```

So `sorted` is working with `[1, 2, 3]` — the numbers, not the counts.

---

### Step 2 — `key=lambda num: freq[num]` assigns a sort score

Before comparing, `sorted` runs this function on each key to get its score:

```python
freq[1]  # → 3
freq[2]  # → 2
freq[3]  # → 1
```

So `1 vs 2 vs 3` aren't compared directly — their **counts** `3 vs 2 vs 1` are.

---

### Step 3 — `reverse=True` sorts highest count first

```python
# ranked by count, descending
[1, 2, 3]  # 1 has count 3, 2 has count 2, 3 has count 1
```

---

### Step 4 — `[:k]` slices the top k items

```python
[1, 2, 3][:2]  # → [1, 2]
```

---

### Full picture

```
keys:             [1,    2,    3]
counts (scores):  [3,    2,    1]
sorted desc:      [1,    2,    3]
[:2]          →   [1,    2]       ← final answer
```

---

> **Core idea:** `sorted(freq, key=lambda num: freq[num])` sorts the **keys**, but ranks them by their **values**.

---
## Reflection

Write your answers in your own words:

1. What is the canonical key trick in Group Anagrams, and why does sorting work as the key?  
=> The key trick is to sort all the characters in a word alphabetically and then compare them to know if they are anagrams or not. Sorting sorts the characters of words alphabetically so whatever their combination may be or order may be, we get the one single key for anagrams.  

2. In Top K Frequent, what are the two distinct steps? What data structure is used in each?  
=> First we create a frequency map, then in second step we convert that dictionaary to sorted list using a lambda function, along with reverse parameter true and slice that for top k frequent elements. We use dictionary for frequency map and list.  

3. You're building a recommendation engine. Users' watch histories are lists of movie genres. How would you use frequency counting to personalize recommendations?  
=> For a user I will calculate the genres and their counts, and I will suggest him/her the genre with the maximum count.I can also rank all genres by frequency, not just suggest the top 1. A genre the user uniquely watches heavily (Nordic Noir) is a stronger signal than a genre everyone watches (Action) — this is the same intuition behind TF-IDF applied to user behavior.    

4. `Counter` and a plain `dict` both work for frequency counting. When would you choose one over the other?  
=> For implementing the concepts from scratch and to understand the underlying mechanism I will choose plain dict. And in production for robustness, ease , I will use Counter.